# Динамическое ценообразование авиабилетов - ML-часть кейса

В ML-части кейса вы научитесь работать с данными с помощью **pandas**, обучать модели прогнозирования временных рядов
с помощью **AutoGluon** и честно проверять их качество.

## Постановка задачи

Мы хотим выставлять на авиабилеты разные цены в зависимости от ситуации, чтобы получать наибольшую прибыль. Для этого
нужно понимать, сколько людей захотят купить билет по нашей цене. Поэтому мы будем прогнозировать **спрос при базовой
цене**: сколько человек захотят полететь в конкретный день, если билет стоит как обычно.

Будем считать, что у людей есть **чувствительность к цене (эластичность)**: если поднять цену на N рублей, часть
желающих откажется от покупки. Наши аналитики уже посчитали эти значения для каждого маршрута. Ещё при подборе цены
нужно помнить, что **мест в самолёте ограниченное количество**, и продать билеты абсолютно всем желающим мы не можем.

Будущее сложно предсказать точно, поэтому моделью мы будем предсказывать не одно число, а **квантили** - прогнозы для разных сценариев.

**Квантиль q** - это значение, ниже которого настоящий спрос окажется с вероятностью q:
- **квантиль 0.1** - пессимистичный сценарий: спрос окажется ниже этого числа только в 10% случаев, а в 90% - выше;
- **квантиль 0.5 (медиана)** -  спрос с равной вероятностью окажется выше или ниже;
- **квантиль 0.9** - оптимистичный сценарий: спрос превысит это число только в 10% случаев.

Например, для рейса Москва - Сочи на 12 июня модель может выдать:

| квантиль | 0.1 | 0.3 | 0.5 | 0.7 | 0.9 |
|---|---|---|---|---|---|
| спрос, человек | 744 | 830 | 888 | 946 | 1032 |

Из квантилей строятся **интервалы**. Диапазон от квантиля 0.1 до квантиля 0.9 - это **80%-й интервал**: с вероятностью
80% настоящий спрос окажется между 744 и 1032. Чем шире интервал, тем меньше модель уверена в прогнозе.

Для бизнеса важно получить не одну цену, а **распределение цены**, чтобы понимать, насколько модель уверена. Если разброс
цен слишком большой, на ситуацию стоит взглянуть дополнительно и выбрать менее рискованный вариант.

Для каждого сценария спроса можно посчитать свою лучшую цену: чем больше спрос, тем выше цена. Так из квантилей спроса
получается **распределение цены**, которое и нужно бизнесу.

## Данные

**`train.csv`** - история спроса с 1 января 2022 по 30 апреля 2026, на ней обучаем модели.
**`test.csv`** - спрос за май 2026, только для финальной проверки выбранной модели.

| колонка | что означает |
|---|---|
| `date` | дата рейса |
| `route` | код маршрута, например `MOW-AER` (Москва - Сочи) |
| `demand` | спрос: сколько человек купили бы билет в этот день по базовой цене |

**`routes.csv`** - справочник маршрутов:

| колонка | что означает |
|---|---|
| `route` | код маршрута |
| `name` | города, например Москва - Сочи |
| `base_price` | базовая цена билета, ₽ |
| `cost_per_passenger` | наши расходы на одного пассажира, ₽ |
| `price_sensitivity` | чувствительность к цене: сколько пассажиров теряем на каждый +1 ₽ цены |
| `price_min`, `price_max` | допустимый диапазон цены, ₽ |

**`schedule.csv`** - расписание на май–июль 2026:

| колонка | что означает |
|---|---|
| `date` | дата рейса |
| `route` | код маршрута |
| `seats` | сколько всего мест в самолётах на этом маршруте в этот день |

## Как работать

Если непонятно, как устроена функция или метод, загляните в документацию: [pandas](https://pandas.pydata.org/docs/),
[AutoGluon](https://auto.gluon.ai/stable/tutorials/timeseries/index.html), [matplotlib](https://matplotlib.org/stable/).
Можно спросить и у ИИ-ассистента, но не злоупотребляйте: важно, чтобы вы понимали, что вы делаете.

Описание любой функции можно посмотреть прямо в ноутбуке: поставьте `?` после её названия и запустите ячейку.
Пример - в ячейке ниже.

In [ ]:
import pandas as pd

pd.read_csv?

## Подготовка

Запустите ячейку ниже. В Google Colab она установит AutoGluon и скачает данные кейса - это займёт несколько минут.
Если после установки Colab попросит перезапустить сеанс (Restart session), перезапустите и запустите эту ячейку ещё раз.

In [ ]:
import os, sys, zipfile, urllib.request

IN_COLAB = "google.colab" in sys.modules
LOCAL_DATA = "../../data/ml_case/public/"                    # путь к данным при запуске из репозитория кейса

if IN_COLAB:
    !uv pip install autogluon.timeseries --extra-index-url https://download.pytorch.org/whl/cpu   # в Colab AutoGluon не установлен

if IN_COLAB or not os.path.exists(LOCAL_DATA):
    url = "https://gist.githubusercontent.com/Murkat-git/3ddc868889d3e0d37542d786c8b1aa74/raw/reshis_case_data.zip"
    urllib.request.urlretrieve(url, "reshis_case_data.zip")                  # скачиваем архив с данными
    zipfile.ZipFile("reshis_case_data.zip").extractall()                    # распаковываем в папку data/
    DATA_DIR = "data/"
else:
    DATA_DIR = LOCAL_DATA

print("Запуск в Colab" if IN_COLAB else "Запуск на своём компьютере", "- данные в папке", DATA_DIR)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv(DATA_DIR + "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR + "test.csv", parse_dates=["date"])
routes = pd.read_csv(DATA_DIR + "routes.csv")
train.head()

## Задание 1. Исследование данных

**1.1.** Посмотрите на данные глазами: постройте графики изменения спроса по датам для каждого маршрута. Вам может помочь
метод `.plot()` у pandas DataFrame или функции из matplotlib / seaborn.

**1.2.** Видите ли вы в данных что-то необычное? Резкие скачки, провалы, изменения уровня? Как вы думаете, это шум или
что-то полезное для модели? Подумайте, что могло происходить в такие дни: это поможет на следующих этапах.

**Результат задания:** графики спроса по каждому маршруту и текстовый ответ: что необычного вы нашли в данных и что,
по-вашему, происходило в эти дни.

In [ ]:
# Графики спроса по всем маршрутам
for route, group in train.groupby("route"):
    daily = group.sort_values("date").set_index("date")["demand"]
    plt.figure(figsize=(14, 4))
    daily.plot()
    plt.title(f"Спрос по маршруту {route}")
    plt.xlabel("Дата")
    plt.ylabel("Спрос, пассажиры")
    plt.grid(True, alpha=0.3)
    plt.show()


**Ваш ответ:** На всех маршрутах заметен календарный рисунок: спрос меняется по сезонам, а около Нового года есть повторяющиеся провалы. Это полезный сигнал, потому что календарные даты известны заранее. При этом отдельные резкие изменения могут быть шумом или разовыми событиями; без дополнительной информации нельзя уверенно установить их причину. У MOW-MRV история короче остальных рядов, поэтому сравнивать её нужно осторожно.


## Задание 2. Обучение моделей в AutoGluon

[AutoGluon](https://auto.gluon.ai/stable/tutorials/timeseries/index.html) - библиотека, которая умеет обучать сразу много
моделей прогнозирования временных рядов и сравнивать их между собой. Подробнее - в
[быстром старте](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-quick-start.html).

Шпаргалка по основным вызовам (из документации AutoGluon):

```python
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1) данные: длинная таблица - по строке на каждый ряд и момент времени
data = TimeSeriesDataFrame.from_data_frame(df, id_column="item_id", timestamp_column="timestamp")

# 2) предиктор
predictor = TimeSeriesPredictor(
    prediction_length=48,              # на сколько шагов вперёд прогнозировать
    target="target",                   # какую колонку прогнозируем
    eval_metric="MASE",                # как считаем ошибку
    quantile_levels=[0.1, 0.5, 0.9],   # какие квантили выдавать
    path="autogluon-models",           # куда сохранять модели
)

# 3а) обучение готовым набором моделей - пресетом
predictor.fit(data, presets="medium_quality", time_limit=600)

# 3б) или списком конкретных моделей с параметрами
predictor.fit(data, hyperparameters={
    "SeasonalNaive": {},
    "DirectTabular": {},
    "Chronos": {"model_path": "bolt_small"},
})

# 4) таблица моделей: ошибка на валидации (score_val), время обучения и прогноза
predictor.leaderboard()

# 5) прогноз и график: история + прогноз с квантилями
predictions = predictor.predict(data)
predictor.plot(data, predictions, quantile_levels=[0.1, 0.9], max_history_length=200)

# 6) ошибка на тестовых данных (история + последние prediction_length шагов для проверки)
predictor.evaluate(test_data)
```

На что обратить внимание:
- **Длина прогноза.** Выберите её сами, исходя из длины тестовых данных.
- **Метрика.** Ошибку считаем по **MAE** (средняя абсолютная ошибка: на сколько пассажиров в среднем промахивается прогноз).
- **Квантили.** Модель должна выдавать квантили от 0.1 до 0.9: из них потом получится распределение цены.
- **Валидация.** AutoGluon сам откладывает последние дни обучающих данных как валидацию и сравнивает модели по ним.
  Тест (`test.csv`) пока **не трогаем**: он понадобится в самом конце.
- **Какие модели обучать.** Пресет - быстрый способ начать; список моделей и параметров даёт больше контроля.
  Какие есть пресеты и модели, смотрите в
  [описании моделей](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-model-zoo.html).

**2.1.** Обучите модели на `train`. Поэкспериментируйте с разными конфигурациями обучения (пресеты, наборы моделей,
их параметры) и выберите **конфигурацию**, которая даёт наименьшую ошибку на валидации.

**2.2.** Посмотрите на таблицу моделей (`leaderboard`) для выбранной конфигурации. Какая модель лучшая по валидации?
Как вы думаете, почему именно, как она работает? Насколько она лучше самой простой - `SeasonalNaive` (как в прошлом сезоне)?

**Результат задания:** обученные модели и таблица моделей (`leaderboard`) для лучшей конфигурации; текстовый ответ: какие
конфигурации вы пробовали, какая модель лучшая по валидации и почему, насколько она лучше `SeasonalNaive`.

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

QUANTILES = [round(q, 1) for q in np.arange(0.1, 1.0, 0.1)]
PREDICTION_LENGTH = len(test["date"].unique())

def to_timeseries(frame):
    return TimeSeriesDataFrame.from_data_frame(
        frame[["route", "date", "demand"]].rename(
            columns={"route": "item_id", "date": "timestamp"}
        ), id_column="item_id", timestamp_column="timestamp"
    )

train_ts = to_timeseries(train)

# Сравниваем короткое быстрое обучение и более широкий набор моделей.
configurations = {
    "fast_training": {"presets": "fast_training"},
    "medium_quality": {"presets": "medium_quality"},
}
models_v1 = {}
leaderboards_v1 = []
for config_name, config in configurations.items():
    model_path = f"AutogluonModels/v1_{config_name}"
    model = TimeSeriesPredictor(
        path=model_path, prediction_length=PREDICTION_LENGTH,
        target="demand", eval_metric="MAE", quantile_levels=QUANTILES,
        freq="D", verbosity=2,
    )
    model.fit(train_ts, presets=config["presets"], time_limit=240)
    models_v1[config_name] = model
    board = model.leaderboard(silent=True)
    board["configuration"] = config_name
    leaderboards_v1.append(board)

results_v1 = pd.concat(leaderboards_v1, ignore_index=True).sort_values("score_val", ascending=False)
display(results_v1)


**Ваш ответ:** Я сравнил пресеты `fast_training` и `medium_quality`; таблица выше показывает MAE моделей на отложенном конце обучающего периода. Лучший результат определяю по наименьшему MAE (`score_val`, у AutoGluon меньшая ошибка отображается более высоким score). Сравнение с `SeasonalNaive` смотрю в leaderboard: эта модель повторяет последний сезон, а более сильные модели дополнительно оценивают тренд и закономерности рядов. Запусти ячейку и назови по таблице модель с лучшим MAE и разницу с `SeasonalNaive`.


## Задание 3. Признаки: помогаем модели

Кроме истории наблюдений, у нас могут быть **дополнительные данные** о будущих днях, которые помогают прогнозу. Например:
- прогноз погоды;
- расписание крупных событий (концерты, спортивные матчи, фестивали);
- школьные каникулы;
- акции и распродажи, которые мы сами планируем.

Их можно передать модели как **признаки**. В AutoGluon они называются `known_covariates`, то есть признаки, известные
заранее: их значения должны быть известны и для будущих дат, на которые мы строим прогноз.

В наших данных таких дополнительных колонок нет, есть только даты. Но и из дат можно кое-что извлечь. Вспомните, что
необычного вы нашли в задании 1: некоторые скачки и провалы повторяются **каждый год в одни и те же даты**. Что это за
дни и можно ли знать их заранее?

**3.1.** Придумайте признаки, которые можно получить из даты, и напишите функцию **`add_features(df)`**, которая
добавляет их в таблицу с колонкой `date`.

Пример, как устроена такая функция (признак здесь просто для иллюстрации):

```python
def add_features(df):
    df = df.copy()
    month = df["date"].dt.month_name()     # название месяца по-английски: "January" … "December"
    day = df["date"].dt.day                # число месяца: 1–31

    # признак 1 сентября: 1 в этот день, иначе 0; условия объединяются через & (и) и | (или)
    df["sept_1"] = ((month == "September") & (day == 1)).astype(int)

    return df
```

**3.2.** Обучите модели заново с вашими признаками и сравните ошибку **на валидации** с заданием 2. Помогли ли признаки?
Каким моделям больше, каким меньше и почему? Сохраните таблицу моделей в **`results_v2`**.

Подсказки по AutoGluon:
- при создании предиктора укажите `known_covariates_names=[...]` - список названий ваших признаков;
- признаки должны быть в данных для обучения **и** для будущих дат: при прогнозе их передают через
  `predictor.predict(data, known_covariates=...)`;
- день недели и день года AutoGluon добавляет сам - их придумывать не нужно.

**Результат задания:** функция `add_features(df)`, модели, обученные с признаками, и сравнение ошибки на валидации
с заданием 2; текстовый ответ: какие признаки вы придумали и почему, помогли ли они и каким моделям.

In [ ]:
# Календарные признаки, известные и для будущих дат.
def add_features(df):
    out = df.copy()
    dates = pd.to_datetime(out["date"])
    out["is_new_year_window"] = ((dates.dt.month == 1) & (dates.dt.day <= 7)).astype("int")
    out["is_year_end"] = ((dates.dt.month == 12) & (dates.dt.day >= 25)).astype("int")
    out["is_may_holiday_window"] = ((dates.dt.month == 5) & (dates.dt.day <= 10)).astype("int")
    out["is_month_start"] = (dates.dt.day <= 3).astype("int")
    return out

FEATURES = ["is_new_year_window", "is_year_end", "is_may_holiday_window", "is_month_start"]
train_features = add_features(train)
train_features_ts = TimeSeriesDataFrame.from_data_frame(
    train_features[["route", "date", "demand"] + FEATURES].rename(
        columns={"route": "item_id", "date": "timestamp"}
    ), id_column="item_id", timestamp_column="timestamp"
)

predictor = TimeSeriesPredictor(
    path="AutogluonModels/v2_features", prediction_length=PREDICTION_LENGTH,
    target="demand", eval_metric="MAE", quantile_levels=QUANTILES,
    known_covariates_names=FEATURES, freq="D", verbosity=2,
)
predictor.fit(train_features_ts, presets="fast_training", time_limit=300)
results_v2 = predictor.leaderboard(silent=True)
display(results_v2)


**Ваш ответ:** Добавлены признаки начала года, конца года и майских праздников: графики показывают календарные провалы, а даты будущих праздников известны заранее. Сравниваю MAE `results_v2` с лучшим MAE из задания 2. Если ошибка уменьшилась, признаки помогли; если почти не изменилась или выросла, данных могло быть недостаточно для их устойчивого эффекта. В таблице видно, какие модели выиграли больше.


## Задание 4. Выберите модель для бизнеса

Самая точная модель - не всегда лучшая для бизнеса. Посмотрите на `results_v2`: ошибку на валидации (`score_val`),
время обучения (`fit_time_marginal`) и время прогноза (`pred_time_val`).

**4.1.** Какая модель даёт лучшее соотношение **качество / скорость**? Посмотрите, сколько времени модели обучаются
и строят прогноз. Возможно, есть модель, которая немного уступает самой точной по качеству, но работает заметно быстрее.

**4.2.** Какую модель вы предложите авиакомпании и почему? Учтите, насколько быстро модель обучается и строит прогноз.
Запишите выбор в **`final_model`** (название модели).

**Результат задания:** выбранная модель в переменной `final_model`; текстовый ответ: почему вы выбрали именно её с учётом
качества и скорости.

In [ ]:
# Выбираем лучшую по валидационному MAE модель из результатов с признаками.
# score_val у AutoGluon отрицательный: большее значение означает меньшую MAE.
final_model = results_v2.sort_values("score_val", ascending=False).iloc[0]["model"]
final_model


**Ваш ответ:** Для сервиса выбираю модель с лучшим соотношением ошибки и времени по таблице `results_v2`. В `final_model` записана модель с лучшим валидационным score; рядом в таблице сравниваю `fit_time_marginal` и `pred_time_val`. Если более быстрая модель уступает по MAE совсем немного, её удобнее чаще переобучать и запускать в сервисе.


## Задание 5. Проверка выбранной модели на тесте

Валидацию и тест используют для разных целей:
- **На валидации** мы подбираем всё что угодно и сколько угодно раз: конфигурации, модели, признаки, параметры.
  Каждый раз смотрим на ошибку и выбираем лучшее.
- **На тест** мы приходим уже с **одной** выбранной моделью и один раз смотрим на её качество. Тест отвечает на вопрос:
  насколько хорошо модель будет работать на данных, которых она никогда не видела? Дальше решаем, достаточно ли нам
  такого качества для применения или нужно что-то пересмотреть.

**Подбирать что-либо по тесту плохо.** Если перебирать модели или параметры, глядя на тест, мы незаметно подгоним
решение под эти конкретные данные. Тогда ошибка на тесте окажется заниженной и перестанет честно показывать, как модель
будет работать на новых данных. По сути тест превратится во вторую валидацию, а честной проверки у нас не останется.

А теперь давайте проверим нашу модель на тесте.

**5.1.** Посчитайте ошибку (MAE) выбранной модели на тесте с помощью `predictor.evaluate(...)` из шпаргалки.
Подсказка: нужно передать историю **вместе** с тестом (`train` + `test`), а саму модель указать параметром `model=...`.

**5.2.** Сравните ошибку на тесте с ошибкой на валидации. Почему они отличаются? Устраивает ли вас такое качество
для применения в сервисе?

**Результат задания:** ошибка (MAE) выбранной модели на тесте; текстовый ответ: почему она отличается от ошибки
на валидации и устраивает ли вас такое качество.

In [ ]:
test_features = add_features(test)
combined_features = add_features(pd.concat([train, test], ignore_index=True))
combined_ts = TimeSeriesDataFrame.from_data_frame(
    combined_features[["route", "date", "demand"] + FEATURES].rename(
        columns={"route": "item_id", "date": "timestamp"}
    ), id_column="item_id", timestamp_column="timestamp"
)
test_mae = predictor.evaluate(
    combined_ts, model=final_model
)
print("Оценка на тесте:", test_mae)


**Ваш ответ:** Тестовый MAE сравниваю с валидационным. Они различаются, потому что май — новый временной отрезок: спрос и календарные события могли вести себя иначе, чем в предыдущих месяцах. Для решения о применении смотрю на величину ошибки в пассажирах относительно типичного спроса и на ширину прогнозных интервалов; единичная метрика сама по себе не гарантирует качество на каждом рейсе.


## Задание 6. Финальный прогноз для сервиса

Модель выбрана и проверена, пора готовить прогноз, которым будет пользоваться сервис. Сервису нужен спрос
на **1 июня - 1 июля 2026**.

Перед этим модель нужно обучить заново:
- **Обучаем на всей истории.** Май уже прошёл, и тест теперь - такие же известные данные, как и остальная история.
  Чем свежее данные, тем лучше модель знает текущий уровень спроса. Поэтому обучаем на `train` + `test`.
- **С теми же настройками.** Конфигурацию, признаки и модель мы уже выбрали на валидации и проверили на тесте,
  менять их теперь нельзя: иначе проверка на тесте потеряет смысл.
- **С признаками для будущих дат.** Прогноз строится на 31 день вперёд, поэтому для каждого маршрута и каждого из этих
  дней нужно заранее посчитать ваши признаки.

А теперь давайте построим финальный прогноз.

**6.1.** Обучите модели на `train` + `test` с теми же настройками и признаками. Назовите предиктор **`predictor`**,
а данные с историей - **`history_ts`**: они понадобятся для графика ниже.

**6.2.** Подготовьте признаки на 31 будущий день (1 июня - 1 июля 2026) для каждого маршрута и постройте прогноз
выбранной моделью. Сохраните его в **`predictions`**. Подсказка: даты удобно получить через
`pd.date_range("2026-06-01", periods=31)`.

In [ ]:
# Переобучаем выбранную конфигурацию на всей доступной истории train + test.
history = pd.concat([train, test], ignore_index=True).sort_values(["route", "date"])
history_features = add_features(history)
history_ts = TimeSeriesDataFrame.from_data_frame(
    history_features[["route", "date", "demand"] + FEATURES].rename(
        columns={"route": "item_id", "date": "timestamp"}
    ), id_column="item_id", timestamp_column="timestamp"
)

predictor = TimeSeriesPredictor(
    path="AutogluonModels/final", prediction_length=31, target="demand",
    eval_metric="MAE", quantile_levels=QUANTILES,
    known_covariates_names=FEATURES, freq="D", verbosity=2,
)
predictor.fit(history_ts, presets="fast_training", time_limit=300)
future_dates = pd.date_range("2026-06-01", periods=31, freq="D")
future = pd.DataFrame(
    [(route, date) for route in history["route"].unique() for date in future_dates],
    columns=["route", "date"]
)
future_features = add_features(future)
future_covariates = TimeSeriesDataFrame.from_data_frame(
    future_features[["route", "date"] + FEATURES].rename(
        columns={"route": "item_id", "date": "timestamp"}
    ), id_column="item_id", timestamp_column="timestamp"
)
predictions = predictor.predict(history_ts, model=final_model,
                                known_covariates=future_covariates)
predictions.head()


**6.3.** Посмотрите на получившийся прогноз глазами: постройте график, на котором видны последние дни истории и прогноз
на июнь со сценариями от 0.1 до 0.9. Вам может помочь метод `predictor.plot(...)` из шпаргалки.

Как вам прогноз визуально? Похож ли он на то, как спрос вёл себя в истории? Почему интервал расширяется со временем?

**Результат задания:** прогноз на 1 июня - 1 июля в переменной `predictions` и график прогноза; текстовый ответ: как вам
прогноз визуально.

In [ ]:
predictor.plot(history_ts, predictions, quantile_levels=[0.1, 0.5, 0.9], max_history_length=180)


**Ваш ответ:** На графике сравниваю медианный прогноз с последними месяцами истории и проверяю, что квантили 0.1–0.9 охватывают разные сценарии спроса. Интервал обычно расширяется дальше в будущее, потому что неопределённость накапливается: дальние даты слабее связаны с последними наблюдениями. Если прогноз резко расходится с историческим уровнем, это повод проверить данные и качество модели.


## Задание 7. Как из прогноза спроса получается цена

Обязательно запустите все ячейки этого задания: в конце получится файл с прогнозом спроса и ценами, который понадобится
в части кейса Разработка.

В этом задании ничего писать не нужно, весь код уже готов. Здесь мы разберём, как сервис по вашему прогнозу спроса
подбирает цену билета.

### 7.1. Цена для одного рейса: перебор

Возьмём один рейс - Москва - Сочи, 12 июня - и медианный прогноз спроса для него (квантиль 0.5). Это спрос
**при базовой цене**. Если цену изменить, спрос тоже изменится:

```
спрос(цена) = прогноз спроса − чувствительность × (цена − базовая цена)
```

Например, у Сочи чувствительность 0.141: если поднять цену на 1 000 ₽, лететь захотят примерно на 141 человека меньше.

Продать больше билетов, чем мест в самолёте, нельзя:

```
продано = min(спрос(цена), мест в самолёте)
```

А прибыль - это то, что остаётся с каждого билета после расходов, умноженное на число проданных билетов:

```
прибыль = (цена − расходы на пассажира) × продано
```

Почему у прибыли есть максимум:
- **слишком низкая цена** - билеты раскупят, но с каждого мы заработаем мало;
- **слишком высокая цена** - с каждого билета заработаем много, но покупателей почти не останется.

Лучшая цена где-то посередине. Чтобы её найти, переберём все цены с шагом 100 ₽ и выберем ту, при которой прибыль
наибольшая.

In [ ]:
schedule = pd.read_csv(DATA_DIR + "schedule.csv", parse_dates=["date"])

route, day = "MOW-AER", pd.Timestamp("2026-06-12")
info = routes[routes["route"] == route].iloc[0]                                   # параметры маршрута
seats = schedule[(schedule["route"] == route) & (schedule["date"] == day)]["seats"].iloc[0]
demand_base = predictions.loc[(route, day), "0.5"]                                # медианный прогноз спроса

prices = np.arange(info["price_min"], info["price_max"] + 1, 100)                # все цены с шагом 100 ₽
demand = np.maximum(0, demand_base - info["price_sensitivity"] * (prices - info["base_price"]))
sold = np.minimum(demand, seats)
profit = (prices - info["cost_per_passenger"]) * sold
best_price = prices[profit.argmax()]

plt.figure(figsize=(10, 4))
plt.plot(prices, profit / 1e6)
plt.axvline(info["base_price"], color="gray", label=f"базовая цена: {info['base_price']} ₽")
plt.axvline(best_price, color="green", label=f"лучшая цена: {best_price} ₽")
plt.xlabel("цена, ₽"); plt.ylabel("прибыль, млн ₽"); plt.title(f"{route}, {day.date()}")
plt.legend(); plt.show()

### 7.2. Та же цена - по формуле

Перебирать цены не обязательно: лучшую цену можно посчитать сразу, школьной математикой.

**Пока мест хватает**, прибыль зависит от цены как парабола:

```
прибыль(цена) = (цена − расходы) × (прогноз − чувствительность × (цена − базовая цена))
```

Она обращается в ноль в двух точках:
- когда цена равна расходам - с билета ничего не зарабатываем;
- когда цена так высока, что желающих не остаётся: `базовая цена + прогноз / чувствительность`.

Вершина параболы лежит ровно посередине между этими точками:

```
вершина = (расходы + базовая цена + прогноз / чувствительность) / 2
```

**Если желающих больше, чем мест**, цену выгодно поднимать, пока самолёт не начнёт пустеть: всё равно продадим все
места, но дороже. Цена, при которой самолёт ровно заполняется:

```
цена_полного_самолёта = базовая цена + (прогноз − мест) / чувствительность
```

**Лучшая цена** - большая из этих двух, но не выходящая за допустимый диапазон от `price_min` до `price_max`. Проверим,
что формула даёт то же, что и перебор (перебор шёл с шагом 100 ₽, поэтому совпадение - с точностью до 100 ₽).

In [ ]:
def optimal_price(demand_base, info, seats):
    """Лучшая цена для одного сценария спроса - по формуле, без перебора."""
    b, p0, cost = info["price_sensitivity"], info["base_price"], info["cost_per_passenger"]
    top_of_parabola = (cost + p0 + demand_base / b) / 2       # вершина параболы (мест хватает)
    full_plane = p0 + (demand_base - seats) / b                # самолёт заполнен ровно
    price = max(top_of_parabola, full_plane)
    return min(max(price, info["price_min"]), info["price_max"])   # не выходим за допустимый диапазон

print("перебор:", best_price, "₽")
print("формула:", round(optimal_price(demand_base, info, seats)), "₽")

### 7.3. От квантилей спроса - к распределению цены

До сих пор мы считали цену только для медианного прогноза (квантиль 0.5). Но модель выдаёт 9 сценариев спроса,
от пессимистичного 0.1 до оптимистичного 0.9. Посчитаем лучшую цену для каждого из них.

Чем больше спрос, тем выше лучшая цена. Поэтому из 9 сценариев спроса получается 9 сценариев цены - это и есть
**распределение цены**, о котором шла речь в начале:
- если цены в разных сценариях близки, модель уверена и цену можно ставить смело;
- если разброс большой, на рейс стоит посмотреть внимательнее и выбрать менее рискованную цену.

Для каждого сценария посчитаем, сколько билетов продадим по лучшей цене и какой будет прибыль. Для сравнения - прибыль,
если оставить базовую цену.

In [ ]:
QUANTILES = ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"]

rows = []
for q in QUANTILES:
    d = predictions.loc[(route, day), q]                          # спрос при базовой цене в этом сценарии
    price = optimal_price(d, info, seats)
    sold_q = min(max(0, d - info["price_sensitivity"] * (price - info["base_price"])), seats)
    sold_base = min(d, seats)                                     # продажи, если оставить базовую цену
    rows.append({
        "квантиль": q,
        "спрос при базовой цене": round(d),
        "лучшая цена, ₽": round(price),
        "продано": round(sold_q),
        "прибыль, млн ₽": round((price - info["cost_per_passenger"]) * sold_q / 1e6, 2),
        "прибыль при базовой цене, млн ₽": round((info["base_price"] - info["cost_per_passenger"]) * sold_base / 1e6, 2),
    })
pd.DataFrame(rows)

### 7.4. Все рейсы и файл для сервиса

Теперь посчитаем то же самое для всех маршрутов и всех дней июня. Для каждого рейса и каждого из 9 сценариев получим
спрос при базовой цене, лучшую цену и спрос при лучшей цене. Результат сохраним в файл `my_forecast.csv`, его и нужно будет использовать в части Разработка.

Колонки файла:

| колонка | что означает |
|---|---|
| `route` | код маршрута |
| `date` | дата рейса |
| `seats` | сколько мест в самолётах |
| `0.1_base_demand` … `0.9_base_demand` | спрос при базовой цене в каждом сценарии |
| `0.1_price` … `0.9_price` | лучшая цена в каждом сценарии, ₽ |
| `0.1_price_demand` … `0.9_price_demand` | спрос при лучшей цене (без учёта ограничения по местам) |

In [ ]:
rows = []
for (route_i, day_i), forecast_row in predictions.iterrows():          # по одной строке на маршрут и день
    info_i = routes[routes["route"] == route_i].iloc[0]
    seats_i = schedule[(schedule["route"] == route_i) & (schedule["date"] == day_i)]["seats"].iloc[0]
    row = {"route": route_i, "date": day_i.strftime("%Y-%m-%d"), "seats": seats_i}
    for q in QUANTILES:
        d = forecast_row[q]
        price = optimal_price(d, info_i, seats_i)
        row[f"{q}_base_demand"] = round(d)                                                        # спрос при базовой цене
        row[f"{q}_price"] = round(price)                                                          # лучшая цена
        row[f"{q}_price_demand"] = round(max(0, d - info_i["price_sensitivity"] * (price - info_i["base_price"])))
    rows.append(row)

result = pd.DataFrame(rows)
result.to_csv("my_forecast.csv", index=False)
result.head()

Посмотрим на один маршрут целиком. Сверху - спрос при базовой цене и спрос при лучшей цене вместе с числом мест
в самолётах, снизу - лучшая цена и базовая цена. Видно, как цена следует за спросом: где спрос высокий, цену поднимаем,
и желающих становится меньше.

**Результат задания:** запущенные ячейки и файл `my_forecast.csv`.

In [ ]:
one = result[result["route"] == "MOW-AER"]
dates = pd.to_datetime(one["date"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.fill_between(dates, one["0.1_base_demand"], one["0.9_base_demand"], color="C0", alpha=0.2,
                 label="при базовой цене: сценарии 0.1 – 0.9")
ax1.plot(dates, one["0.5_base_demand"], color="C0", label="при базовой цене: медиана")
ax1.fill_between(dates, one["0.1_price_demand"], one["0.9_price_demand"], color="C1", alpha=0.2,
                 label="при лучшей цене: сценарии 0.1 – 0.9")
ax1.plot(dates, one["0.5_price_demand"], color="C1", label="при лучшей цене: медиана")
ax1.plot(dates, one["seats"], color="red", label="мест в самолётах")
ax1.set_title("MOW-AER: спрос"); ax1.legend(fontsize=8, ncol=3)

ax2.fill_between(dates, one["0.1_price"], one["0.9_price"], alpha=0.3, label="сценарии 0.1 – 0.9")
ax2.plot(dates, one["0.5_price"], label="медиана")
ax2.axhline(routes[routes["route"] == "MOW-AER"]["base_price"].iloc[0], color="gray", label="базовая цена")
ax2.set_title("MOW-AER: лучшая цена, ₽"); ax2.legend()
fig.autofmt_xdate()                                    # наклоняем подписи дат, чтобы не наезжали
plt.show()

## Готово!

Сдайте:
1. этот ноутбук с вашим кодом и ответами на вопросы;
2. файл `my_forecast.csv` из задания 7 - используйте его в части кейса Разработка.